In [1]:
import os
import glob
import pandas as pd
import anndata as ad
from datetime import datetime
from fcsparser import parse

# ============================================================
# 输入目录（两个来源）
# ============================================================

input_dirs = [
    "/public/users/xueyupeng/VZV/cytof/subtype_raw/",
    "/public/users/xueyupeng/VZV/lvzhu/subtype_raw/"
]

# 输出目录
output_base = "/public/users/xueyupeng/VZV/cytof/subtype_raw_combined"
os.makedirs(output_base, exist_ok=True)



In [2]:
# ============================================================
# 日志文件
# ============================================================

log_file = os.path.join(output_base, "combine_log.txt")

def log_print(*args):

    msg = " ".join(str(a) for a in args)

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    final_msg = f"[{timestamp}] {msg}"

    print(final_msg)

    with open(log_file, "a") as f:
        f.write(final_msg + "\n")

In [3]:
# ============================================================
# 读取 FCS
# ============================================================

def read_fcs(file):
    meta, data = parse(file, reformat_meta=True)
    return data

In [4]:
# ============================================================
# 处理单个 subtype 文件夹
# ============================================================

def process_one_folder(folder_paths):

    all_files = []

    # 收集两个目录下所有 fcs
    for folder_path in folder_paths:

        files = glob.glob(os.path.join(folder_path, "*.fcs"))

        all_files.extend(files)

    if len(all_files) == 0:
        return None

    dfs = []

    for f in all_files:

        try:

            df = read_fcs(f)

            # ====================================================
            # 样本名（避免重名）
            # ====================================================

            parent1 = os.path.basename(os.path.dirname(os.path.dirname(f)))
            # cytof 或 lvzhu

            sample_name = os.path.basename(f).replace(".fcs", "")

            # 加来源前缀
            sample_id = f"{parent1}_{sample_name}"

            df["sample_id"] = sample_id

            dfs.append(df)

            log_print("Loaded:", f)

        except Exception as e:

            log_print("Failed:", f)
            log_print("Error:", str(e))

    if len(dfs) == 0:
        return None
    # ============================================================
    # 合并
    # ============================================================

    df_all = pd.concat(dfs, axis=0, ignore_index=True)

    feature_cols = [c for c in df_all.columns if c != "sample_id"]

    # ============================================================
    # 构建 AnnData
    # ============================================================

    X = df_all[feature_cols].values

    var = pd.DataFrame(index=feature_cols)

    obs = pd.DataFrame(
        index=[f"cell_{i}" for i in range(df_all.shape[0])]
    )

    obs["sample_id"] = df_all["sample_id"].values

    adata = ad.AnnData(
        X=X,
        obs=obs,
        var=var
    )

    return adata


In [5]:

# ============================================================
# 清空旧日志
# ============================================================

with open(log_file, "w") as f:
    f.write("===== FCS Combine Log =====\n")

log_print("Start processing")

[2026-05-08 18:42:16] Start processing


In [6]:
# ============================================================
# 获取两个目录共有 subtype
# ============================================================

all_subdirs = set()

for input_base in input_dirs:

    subdirs = [
        d for d in os.listdir(input_base)
        if os.path.isdir(os.path.join(input_base, d))
    ]

    all_subdirs.update(subdirs)

all_subdirs = sorted(all_subdirs)

log_print("Found subtype folders:", len(all_subdirs))

[2026-05-08 18:42:16] Found subtype folders: 25


In [7]:
# ============================================================
# 主循环
# ============================================================

for subdir in all_subdirs:

    folder_paths = []

    # 查找两个目录中是否存在该 subtype
    for input_base in input_dirs:

        folder_path = os.path.join(input_base, subdir)

        if os.path.isdir(folder_path):
            folder_paths.append(folder_path)

    if len(folder_paths) == 0:
        continue

    log_print("")
    log_print("==============================")
    log_print("Processing:", subdir)
    log_print("==============================")

    adata = process_one_folder(folder_paths)

    if adata is None:

        log_print("No valid FCS found.")
        continue

    out_file = os.path.join(output_base, f"{subdir}.h5ad")

    adata.write(out_file)

    log_print("Saved:", out_file)
    log_print("Cells:", adata.n_obs)
    log_print("Markers:", adata.n_vars)
log_print("All done.")

[2026-05-08 18:42:17] 
[2026-05-08 18:42:17] ==============================
[2026-05-08 18:42:17] Processing: B
[2026-05-08 18:42:17] ==============================
[2026-05-08 18:42:17] Loaded: /public/users/xueyupeng/VZV/cytof/subtype_raw/B/V5-L116_20251127-4_7-3.fcs
[2026-05-08 18:42:17] Loaded: /public/users/xueyupeng/VZV/cytof/subtype_raw/B/V6-L173_20251230-3_5-3.fcs
[2026-05-08 18:42:17] Loaded: /public/users/xueyupeng/VZV/cytof/subtype_raw/B/V0-L174_20251230-3_4-3.fcs
[2026-05-08 18:42:17] Loaded: /public/users/xueyupeng/VZV/cytof/subtype_raw/B/V5-L177_20251230-4_3-3.fcs
[2026-05-08 18:42:17] Loaded: /public/users/xueyupeng/VZV/cytof/subtype_raw/B/V2-G165_20251124-3_4-3.fcs
[2026-05-08 18:42:17] Loaded: /public/users/xueyupeng/VZV/cytof/subtype_raw/B/V5-L124_20260107-2_3-3.fcs
[2026-05-08 18:42:17] Loaded: /public/users/xueyupeng/VZV/cytof/subtype_raw/B/V1-G154_20260109-3_1-3.fcs
[2026-05-08 18:42:17] Loaded: /public/users/xueyupeng/VZV/cytof/subtype_raw/B/V5-L174_20251230-3_7-3

# add metadata 

In [1]:
import os
import re
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

In [2]:
# =========================================================
# 读取函数
# =========================================================

def load_cytof_h5ad(
    file_path,
    do_arcsinh=False,
    cofactor=5,
    add_metadata=True
):

    adata = sc.read_h5ad(file_path)

    print(f"\nLoaded: {file_path}")
    print(adata)

    adata.obs["sample_id"] = adata.obs["sample_id"].str.replace(
        "_raw", "", regex=False
    )

    # =====================================================
    # metadata
    # =====================================================

    if add_metadata:

        if "sample_id" not in adata.obs.columns:
            raise ValueError("obs 中缺少 sample_id 列")

        sid = adata.obs["sample_id"].astype(str)

        def infer_group(x):

            if re.search(r"subtype_A", x):
                return "A"

            if re.search(r"-L", x):
                return "L"

            if re.search(r"subtype_V[0-9]+-G", x):
                return "G"

            return None

        adata.obs["Group"] = sid.apply(infer_group)

        print("\nGroup distribution:")
        print(adata.obs["Group"].value_counts())

        # =====================================================
        # split
        # =====================================================

        split_dash = adata.obs["sample_id"].str.split(
            "-",
            expand=True
        )

        adata.obs["part1"] = split_dash[0]
        adata.obs["part2"] = split_dash[1]
        adata.obs["part3"] = split_dash[2]

        # =====================================================
        # Group A
        # =====================================================

        mask_A = adata.obs["Group"] == "A"

        adata.obs.loc[mask_A, "Participant"] = (
            adata.obs.loc[mask_A, "part1"]
        )

        adata.obs.loc[mask_A, "Timepoint"] = (
            adata.obs.loc[mask_A, "part2"]
            .str.split("_")
            .str[0]
        )

        adata.obs.loc[mask_A, "Exp_id"] = (
            adata.obs.loc[mask_A, "part2"]
            .str.split("_")
            .str[1]
        )

        adata.obs.loc[mask_A, "Exp_position"] = (
            adata.obs.loc[mask_A, "part3"]
        )

        # =====================================================
        # Group G/L
        # =====================================================

        mask_GL = adata.obs["Group"].isin(["G", "L"])

        adata.obs.loc[mask_GL, "Timepoint"] = (
            adata.obs.loc[mask_GL, "part1"]
        )

        adata.obs.loc[mask_GL, "Participant"] = (
            adata.obs.loc[mask_GL, "part2"]
            .str.split("_")
            .str[0]
        )

        adata.obs.loc[mask_GL, "Exp_id"] = (
            adata.obs.loc[mask_GL, "part2"]
            .str.split("_")
            .str[1]
        )

        adata.obs.loc[mask_GL, "Exp_position"] = (
            adata.obs.loc[mask_GL, "part3"]
        )

        # =====================================================
        # cleanup
        # =====================================================

        adata.obs.drop(
            columns=["part1", "part2", "part3"],
            inplace=True
        )

        adata.obs["Participant"] = (
            adata.obs["Participant"]
            .str.replace("^subtype_", "", regex=True)
        )

        adata.obs["Timepoint"] = (
            adata.obs["Timepoint"]
            .str.replace("^subtype_", "", regex=True)
        )

        # =====================================================
        # timepoint mapping
        # =====================================================

        def map_timepoint(row):

            grp = row["Group"]
            tp = row["Timepoint"]

            # ===== Group G =====
            if grp == "G" and tp == "V1":
                return "V1Day0"

            elif grp == "G" and tp == "V2":
                return "V1Day1"

            # ===== Group L =====
            elif grp == "L" and tp == "V0":
                return "V1Day0"

            elif grp == "L" and tp == "V1":
                return "V1Day1"

            elif grp == "L" and tp == "V5":
                return "V2Day0"

            elif grp == "L" and tp == "V6":
                return "V2Day1"

            # ===== Group A =====
            elif grp == "A" and tp == "V0":
                return "V1Day0"

            elif grp == "A" and tp == "V1":
                return "V1Day1"

            elif grp == "A" and tp == "V3":
                return "V2Day0"

            elif grp == "A" and tp == "V4":
                return "V2Day1"

            elif grp == "A" and tp == "V6":
                return "V1Day0"

            elif grp == "A" and tp == "V7":
                return "V1Day1"

            elif grp == "A" and tp == "V8":
                return "V2Day0"

            elif grp == "A" and tp == "V9":
                return "V2Day1"

            return None

        adata.obs["Timepoint_V"] = adata.obs.apply(
            map_timepoint,
            axis=1
        )

        adata.obs["GT"] = (
            adata.obs["Group"].astype(str)
            + "_"
            + adata.obs["Timepoint_V"].astype(str)
        )

        print("Metadata parsing finished")

    # =====================================================
    # arcsinh
    # =====================================================

    if do_arcsinh:

        if "raw" not in adata.layers:
            adata.layers["raw"] = adata.X.copy()
            print("Raw layer saved")

        adata.X = np.arcsinh(adata.X / cofactor)

        print(f"arcsinh finished (cofactor={cofactor})")

    return adata


In [ ]:
import os
import scanpy as sc

# =========================================================
# 输入输出目录
# =========================================================

input_dir = "/public/users/xueyupeng/VZV/cytof/subtype_raw_combined/"
output_dir = "/public/users/xueyupeng/VZV/cytof/subtype_raw_combined_metedata/"

os.makedirs(output_dir, exist_ok=True)

# =========================================================
# 获取所有 h5ad
# =========================================================

all_files = sorted([
    f for f in os.listdir(input_dir)
    if f.endswith(".h5ad")
])

print(f"Found {len(all_files)} h5ad files")

# =========================================================
# 主循环
# =========================================================

for file in all_files:

    try:

        print("\n================================================")
        print(f"Processing: {file}")
        print("================================================")

        input_path = os.path.join(input_dir, file)

        # =================================================
        # load + metadata
        # =================================================

        adata = load_cytof_h5ad(
            input_path,
            do_arcsinh=False,
            cofactor=5,
            add_metadata=True
        )

        # =================================================
        # 保存
        # =================================================

        output_path = os.path.join(output_dir, file)

        adata.write(output_path)

        print(f"✓ Saved: {output_path}")

        # =================================================
        # 简单检查
        # =================================================

        print(adata)

        print("\nobs columns:")
        print(list(adata.obs.columns))

        print("\nGroup:")
        print(adata.obs["Group"].value_counts(dropna=False))

        print("\nTimepoint_V:")
        print(adata.obs["Timepoint_V"].value_counts(dropna=False))

    except Exception as e:

        print(f"\nERROR in {file}")
        print(e)

print("\n✅ ALL DONE")

Found 25 h5ad files

Processing: B.h5ad

Loaded: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined/B.h5ad
AnnData object with n_obs × n_vars = 11613270 × 52
    obs: 'sample_id'

Group distribution:
Group
A    6176576
L    3634890
G    1801804
Name: count, dtype: int64
Metadata parsing finished
✓ Saved: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined_metedata/B.h5ad
AnnData object with n_obs × n_vars = 11613270 × 52
    obs: 'sample_id', 'Group', 'Participant', 'Timepoint', 'Exp_id', 'Exp_position', 'Timepoint_V', 'GT'

obs columns:
['sample_id', 'Group', 'Participant', 'Timepoint', 'Exp_id', 'Exp_position', 'Timepoint_V', 'GT']

Group:
Group
A    6176576
L    3634890
G    1801804
Name: count, dtype: int64

Timepoint_V:
Timepoint_V
V1Day0    3431271
V1Day1    3321840
V2Day0    2503486
V2Day1    2356673
Name: count, dtype: int64

Processing: CD4T.h5ad

Loaded: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined/CD4T.h5ad
AnnData object with n_obs × n_vars = 43071717 × 52


# 检测文件数量

In [11]:
import os
import pandas as pd

dirs = [
    "/public/users/xueyupeng/VZV/lvzhu/subtype/",
    "/public/users/xueyupeng/VZV/lvzhu/subtype_raw/"
]

results = []

for base_dir in dirs:

    print(f"\n===== {base_dir} =====")

    if not os.path.exists(base_dir):
        print("Directory not found")
        continue

    for subdir in sorted(os.listdir(base_dir)):

        subdir_path = os.path.join(base_dir, subdir)

        if os.path.isdir(subdir_path):

            file_count = sum(
                os.path.isfile(os.path.join(subdir_path, f))
                for f in os.listdir(subdir_path)
            )

            print(f"{subdir}: {file_count}")

            results.append({
                "base_dir": base_dir,
                "subdir": subdir,
                "file_count": file_count
            })

# 如果想保存成 dataframe
df_counts = pd.DataFrame(results)

print("\n===== Summary =====")
print(df_counts)


===== /public/users/xueyupeng/VZV/lvzhu/subtype/ =====
B: 200
CD4T: 200
CD4TCM: 200
CD4TEM: 200
CD4TEMRA: 200
CD4TN: 200
CD8T: 200
CD8TCM: 200
CD8TEM: 200
CD8TEMRA: 200
CD8TN: 200
Classical_Mono: 200
DC: 200
Intermediate_Mono: 200
NK: 200
NKT: 200
NKbright: 200
NKdim: 200
Non_classical_Mono: 200
Plasmablast: 200
T: 200
Toltal_T: 200
cDCs: 200
gdT: 200
pDCs: 200

===== /public/users/xueyupeng/VZV/lvzhu/subtype_raw/ =====
B: 200
CD4T: 200
CD4TCM: 200
CD4TEM: 200
CD4TEMRA: 200
CD4TN: 200
CD8T: 200
CD8TCM: 200
CD8TEM: 200
CD8TEMRA: 200
CD8TN: 200
Classical_Mono: 200
DC: 200
Intermediate_Mono: 200
NK: 200
NKT: 200
NKbright: 200
NKdim: 200
Non_classical_Mono: 200
Plasmablast: 200
T: 200
Toltal_T: 200
cDCs: 200
gdT: 200
pDCs: 200

===== Summary =====
                                          base_dir              subdir  \
0       /public/users/xueyupeng/VZV/lvzhu/subtype/                   B   
1       /public/users/xueyupeng/VZV/lvzhu/subtype/                CD4T   
2       /public/users/x

In [10]:
import os

raw_dir = "/public/users/xueyupeng/VZV/lvzhu/subtype_raw/"

# =========================
# 遍历所有子目录
# =========================
for subfolder in os.listdir(raw_dir):

    subdir = os.path.join(raw_dir, subfolder)

    if not os.path.isdir(subdir):
        continue

    print(f"\nProcessing: {subfolder}")

    removed = 0

    for fname in os.listdir(subdir):

        # 文件名包含 export_
        if "export_" in fname:

            file_path = os.path.join(subdir, fname)

            try:
                os.remove(file_path)
                removed += 1
                print("Removed:", fname)

            except Exception as e:
                print("❌ Failed:", fname, e)

    print(f"✅ Removed {removed} files")

print("\n🎉 Done!")


Processing: CD8TCM
Removed: export_20260129-3_4-3.fcs_267021.fcs
Removed: export_20260129-4_1-3.fcs_402017.fcs
Removed: export_20260129-3_1-3.fcs_374421.fcs
Removed: export_20260129-4_4-3.fcs_308515.fcs
Removed: export_20260129-3_6-3.fcs_281070.fcs
Removed: export_20260129-2_1-3.fcs_180067.fcs
Removed: export_20260129-2_4-3.fcs_338771.fcs
Removed: export_20260129-2_3-3.fcs_279570.fcs
Removed: export_20260129-4_6-3.fcs_135422.fcs
Removed: export_20260129-1_7-3.fcs_314443.fcs
Removed: export_20260129-3_7-3.fcs_237237.fcs
Removed: export_20260129-3_2-3.fcs_299671.fcs
Removed: export_20260129-4_8-3.fcs_232899.fcs
Removed: export_20260129-4_2-3.fcs_264652.fcs
Removed: export_20260129-1_1-3.fcs_330366.fcs
Removed: export_20260129-4_7-3.fcs_466386.fcs
Removed: export_20260129-1_4-3.fcs_385178.fcs
Removed: export_20260129-3_8-3.fcs_294987.fcs
Removed: export_20260129-2_6-3.fcs_320183.fcs
Removed: export_20260129-1_8-3.fcs_360791.fcs
Removed: export_20260129-1_2-3.fcs_359800.fcs
Removed: expor